# z017 - ¿La mejora de z012 es real o sobreajuste a junio?

z012 (hessian ~100, pocas hojas, ~3000 árboles) dio +35 M en validación, pero todo se eligió y midió en **202106**
(Optuna, revalidación y early stopping). En el público (submit 13) no se vio la mejora.

Prueba en un mes que **ninguno vio**, **202105**:
- **A:** train 202103 → valid 202105 (2 meses de distancia; ojo: 202103 no tiene lags, son todos NaN).
- **B:** train 202104 → valid 202105 (1 mes de distancia, con lags).

Se comparan el **trial 34 de z005** y el **trial 26 de z012**, con los árboles fijos de cada uno, 5 semillas.
Ojo: en 202105 `ccajas_depositos` es NA (variable rota).

In [1]:
# --- 1. Librerías y parámetros comunes ---
import numpy as np
import pandas as pd
import lightgbm as lgb

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
MES_VALID = 202106
SEMILLAS  = [501419, 623353, 786349, 834569, 946331]
N_PARTICIONES = 50        # Shuffle Split del test: particiones al azar del 50% de 202106

# Trial 34 de z005 (el mejor en promedio de 5 semillas)
params = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
    "num_leaves": 137, "min_sum_hessian_in_leaf": 9.755574,
}
ARBOLES = 538

In [2]:
# --- 2. Cargar datos y lags (igual que z003) ---
df = pd.read_parquet("../competencia_01.parquet")
df["clase_binaria"] = (df["clase_ternaria"] == "BAJA+2").astype(int)

no_usar = ["foto_mes", "clase_ternaria", "clase_binaria", "clase_binaria2"]
cols = [c for c in df.columns if c not in no_usar and c != "numero_de_cliente"]

df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
grupo = df.groupby("numero_de_cliente")[cols]
nuevas = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]
df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo

variables = [c for c in df.columns if c not in no_usar]
valid = df[df["foto_mes"] == MES_VALID]
# La ganancia SIEMPRE se calcula con BAJA+2 (aunque se entrene con otro target)
ganancia_cliente = np.where(valid["clase_ternaria"].values == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO)
print("Variables:", len(variables))

Variables: 761


In [3]:
# --- 3. Configuraciones y evaluación ---
fijos = {k: v for k, v in params.items() if k not in ("num_leaves", "min_sum_hessian_in_leaf")}
configs = {
    "z005 trial 34 (hojas 137, hessian 9,76, 538 árboles)":   ({"num_leaves": 137, "min_sum_hessian_in_leaf": 9.755574}, 538),
    "z012 trial 26 (hojas 29, hessian 115,0, 2929 árboles)":  ({"num_leaves": 29, "min_sum_hessian_in_leaf": 115.00875854972075}, 2929),
}
esquemas = {"A: train 202103 -> valid 202105": 202103, "B: train 202104 -> valid 202105": 202104}

valid = df[df["foto_mes"] == 202105]
g_cliente = np.where(valid["clase_ternaria"].values == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO)

def ganancias(prob):
    acum = np.cumsum(g_cliente[np.argsort(prob)[::-1]])
    n = int(np.argmax(acum)) + 1
    return acum[n - 1] / 1e6, n, acum[11_999] / 1e6

filas = []
for esquema, mes_train in esquemas.items():
    train = df[df["foto_mes"] == mes_train]
    datos = lgb.Dataset(train[variables], label=train["clase_binaria"], free_raw_data=False)
    for nombre, (p, arboles) in configs.items():
        for s in SEMILLAS:
            prob = lgb.train({**fijos, **p, "seed": s}, datos, num_boost_round=arboles).predict(valid[variables])
            g, n, g12 = ganancias(prob)
            filas.append({"esquema": esquema, "config": nombre, "semilla": s,
                          "ganancia_max_M": round(g, 1), "mejor_n": n, "top12000_M": round(g12, 1)})
        print(esquema, "|", nombre, "listo", flush=True)
res = pd.DataFrame(filas)

A: train 202103 -> valid 202105 | z005 trial 34 (hojas 137, hessian 9,76, 538 árboles) listo


A: train 202103 -> valid 202105 | z012 trial 26 (hojas 29, hessian 115,0, 2929 árboles) listo


B: train 202104 -> valid 202105 | z005 trial 34 (hojas 137, hessian 9,76, 538 árboles) listo


B: train 202104 -> valid 202105 | z012 trial 26 (hojas 29, hessian 115,0, 2929 árboles) listo


In [4]:
# --- 4. Comparación ---
tabla = res.groupby(["esquema", "config"])[["ganancia_max_M", "top12000_M", "mejor_n"]].agg(["mean", "std"]).round(1)
for esquema in esquemas:
    g = res[res["esquema"] == esquema].pivot(index="semilla", columns="config", values="ganancia_max_M")
    dif = g.iloc[:, 1] - g.iloc[:, 0] if "z012" in g.columns[1] else g.iloc[:, 0] - g.iloc[:, 1]
    print(f"{esquema}: z012 - z005 = {dif.mean():+.1f} M, z012 gana en {(dif > 0).sum()} de {len(dif)} semillas")
tabla

A: train 202103 -> valid 202105: z012 - z005 = -11.9 M, z012 gana en 0 de 5 semillas
B: train 202104 -> valid 202105: z012 - z005 = -16.4 M, z012 gana en 0 de 5 semillas


ganancia_max_M  \
                                                                                             mean   
esquema                         config                                                              
A: train 202103 -> valid 202105 z005 trial 34 (hojas 137, hessian 9,76, 538 árb...          252.6   
                                z012 trial 26 (hojas 29, hessian 115,0, 2929 ár...          240.7   
B: train 202104 -> valid 202105 z005 trial 34 (hojas 137, hessian 9,76, 538 árb...          263.2   
                                z012 trial 26 (hojas 29, hessian 115,0, 2929 ár...          246.8   

                                                                                         \
                                                                                    std   
esquema                         config                                                    
A: train 202103 -> valid 202105 z005 trial 34 (hojas 137, hessian 9,76, 538 árb...  1.7   
                                z012 trial 26 (hojas 29, hessian 115,0, 2929 ár...  1.2   
B: train 202104 -> valid 202105 z005 trial 34 (hojas 137, hessian 9,76, 538 árb...  1.6   
                                z012 trial 26 (hojas 29, hessian 115,0, 2929 ár...  1.3   

                                                                                   top12000_M  \
                                                                                         mean   
esquema                         config                                                          
A: train 202103 -> valid 202105 z005 trial 34 (hojas 137, hessian 9,76, 538 árb...      229.7   
                                z012 trial 26 (hojas 29, hessian 115,0, 2929 ár...      225.1   
B: train 202104 -> valid 202105 z005 trial 34 (hojas 137, hessian 9,76, 538 árb...      240.5   
                                z012 trial 26 (hojas 29, hessian 115,0, 2929 ár...      235.2   

                                                                                         \
                                                                                    std   
esquema                         config                                                    
A: train 202103 -> valid 202105 z005 trial 34 (hojas 137, hessian 9,76, 538 árb...  2.8   
                                z012 trial 26 (hojas 29, hessian 115,0, 2929 ár...  3.7   
B: train 202104 -> valid 202105 z005 trial 34 (hojas 137, hessian 9,76, 538 árb...  2.5   
                                z012 trial 26 (hojas 29, hessian 115,0, 2929 ár...  2.4   

                                                                                   mejor_n  \
                                                                                      mean   
esquema                         config                                                       
A: train 202103 -> valid 202105 z005 trial 34 (hojas 137, hessian 9,76, 538 árb...  7861.4   
                                z012 trial 26 (hojas 29, hessian 115,0, 2929 ár...  7710.4   
B: train 202104 -> valid 202105 z005 trial 34 (hojas 137, hessian 9,76, 538 árb...  8549.6   
                                z012 trial 26 (hojas 29, hessian 115,0, 2929 ár...  9433.0   

                                                                                            
                                                                                       std  
esquema                         config                                                      
A: train 202103 -> valid 202105 z005 trial 34 (hojas 137, hessian 9,76, 538 árb...  1122.2  
                                z012 trial 26 (hojas 29, hessian 115,0, 2929 ár...   381.5  
B: train 202104 -> valid 202105 z005 trial 34 (hojas 137, hessian 9,76, 538 árb...   817.2  
                                z012 trial 26 (hojas 29, hessian 115,0, 2929 ár...   297.8

In [5]:
# --- 5. Anotar ---
ruta_res = "resultados_experimentos.csv"
r = pd.read_csv(ruta_res, sep=";", decimal=",", encoding="utf-8-sig")
r = r[~r["descripcion"].str.startswith("z017")]
for (esquema, config), x in res.groupby(["esquema", "config"]):
    r = pd.concat([r, pd.DataFrame([{
        "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
        "descripcion": f"z017 otro corte {esquema} | {config} (prom. 5 semillas)",
        "meses_train": str([esquemas[esquema]]), "mes_valid": 202105, "semilla": str(SEMILLAS),
        "mejor_n": int(x["mejor_n"].mean()), "ganancia_max_M": round(x["ganancia_max_M"].mean(), 1),
        "ganancia_top12000_M": round(x["top12000_M"].mean(), 1)}])], ignore_index=True)
r["n_enviados"] = r["n_enviados"].astype("Int64")
r.to_csv(ruta_res, sep=";", decimal=",", index=False, encoding="utf-8-sig")
print("Anotado")

Anotado


## Conclusiones (08/10)
| Esquema | z005 trial 34 | z012 trial 26 | z012 − z005 |
|---|---|---|---|
| A: train 202103 → valid 202105 | 252,6 ± 1,7 | 240,7 ± 1,2 | **−11,9 M** (0/5 semillas) |
| B: train 202104 → valid 202105 | 263,2 ± 1,6 | 246,8 ± 1,3 | **−16,4 M** (0/5 semillas) |

- **La mejora de z012 era sobreajuste a 202106**: en un mes que ninguno vio, z005 gana en todas las semillas.
- Coincide con el público: 11 y 12 (z005) dieron 99,11 y 94,82; el 13 (z012) dio 93,20.
- Lección: todo lo que se elige mirando 202106 (Optuna, revalidación, early stopping) hay que confirmarlo en otro mes.
  Lo mismo aplica a z009 (juntar bajas, +22 M en 202106): verificar en 202105 antes de adoptarlo.
- Decisión sugerida: volver a los hiperparámetros de z005 para la entrega final.